# Восстановление пунктуации, регистра и абзацев (русский язык)

**Второе звено пайплайна SpeechToText:**

```
звук → [Whisper] → текст без пунктуации → [эта модель] → текст с пунктуацией
```

Модель **чисто текстовая** — на вход текст без знаков и регистра, на выходе текст с расставленными:
запятыми `,`  точками `.`  вопросами `?`  восклицаниями `!`  многоточиями `…`  **двоеточиями `:`**,
а также с **капитализацией** (регистр слов) и **абзацами** (красная строка). Паузы из звука **не используются**.

### Архитектура
Задача решается как **token-level классификация** с **тремя головами** на каждое слово:
- **PUNCT** — знак после слова (`O / COMMA / PERIOD / QUESTION / EXCLAM / ELLIPSIS / COLON`);
- **CASE** — регистр слова (`LOWER / UPPER_FIRST / UPPER_ALL`);
- **PARA** — начинается ли с этого слова новый абзац (`NO_PARAGRAPH / PARAGRAPH`).

### Модели
| Модель | Файл | Тип |
|---|---|---|
| BiLSTM | `models/lstm_model.py` | baseline (с нуля) |
| Transformer-энкодер | `models/transformer_model.py` | baseline-трансформер (с нуля) |
| ruBERT-tiny2 | `models/pretrained_model.py` | предобученная (`cointegrated/rubert-tiny2`) |
| ruBERT-base | `models/pretrained_model.py` | предобученная (`DeepPavlov/rubert-base-cased`) |

Демо обучается на встроенном **синтетическом мини-корпусе** (офлайн). Для FLEURS — `source="fleurs"`.


## 0. Импорты и настройка

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import torch
from functools import partial

from data.dataset import load_texts
from data.vocab import WordVocab, PunctDataset, collate_fn
from models.lstm_model import BiLSTMPunctuator
from models.transformer_model import TransformerPunctuator
from utils.trainer import train_model
from utils.inference import PunctuationRestorer
from utils.stt_pipeline import STTPunctuationPipeline

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Данные

`source="synthetic"` — офлайн-корпус (в нём встречаются все знаки, аббревиатуры и абзацы).
Чтобы обучать на **Google FLEURS (ru_ru)**, поставьте `SOURCE = "fleurs"` (нужен интернет и `datasets`).

In [ ]:
SOURCE = "synthetic"   # или "fleurs"

texts = load_texts(source=SOURCE, repeat=6)   # repeat дублирует корпус для демо
print(f"Текстов для обучения: {len(texts)}")
print("Пример исходного текста:\n")
print(texts[0])

Как выглядит **вход** модели (то, что отдаёт Whisper) и **целевая** разметка:

In [ ]:
from utils.preprocess import text_to_labeled, degrade_text

sample = texts[0]
toks, punct, case, para = text_to_labeled(sample)
print("ВХОД модели (degraded):")
print(" ", degrade_text(sample))
print("\nЦелевые метки первых 8 слов:")
for w, p, c, a in list(zip(toks, punct, case, para))[:8]:
    print(f"  {w:12} punct={p:9} case={c:11} para={a}")

## 2. Baseline #1 — BiLSTM

Обучаемые эмбеддинги слов + двунаправленная LSTM + три линейные головы.

In [ ]:
vocab = WordVocab.build(texts)
print("Размер словаря:", len(vocab))

ds_word = PunctDataset(texts, vocab)
cfn_word = partial(collate_fn, pad_id=vocab.pad_id)

lstm = BiLSTMPunctuator(vocab_size=len(vocab), pad_id=vocab.pad_id)
print(f"Параметров: {sum(p.numel() for p in lstm.parameters()):,}")
hist_lstm = train_model(lstm, ds_word, cfn_word, epochs=12, batch_size=8, lr=3e-3, device=DEVICE)

## 3. Baseline #2 — Transformer-энкодер (с нуля)

Эмбеддинги слов + позиционные эмбеддинги + `TransformerEncoder`. Предобучения нет.

In [ ]:
transformer = TransformerPunctuator(vocab_size=len(vocab), pad_id=vocab.pad_id)
print(f"Параметров: {sum(p.numel() for p in transformer.parameters()):,}")
hist_tr = train_model(transformer, ds_word, cfn_word, epochs=12, batch_size=8, lr=1e-3, device=DEVICE)

## 4. Предобученные модели — ruBERT-tiny2 и ruBERT-base

⚠️ Требуется интернет: модели скачиваются с HuggingFace при первом запуске.
ruBERT режет слова на subword-токены — метка ставится на **первый** subword слова
(`PretrainedDataset` делает это выравнивание автоматически).

`VARIANT = "tiny"` → `cointegrated/rubert-tiny2` (лёгкая, быстрая).
`VARIANT = "base"` → `DeepPavlov/rubert-base-cased` (тяжелее, точнее).

In [ ]:
from models.pretrained_model import (
    RuBertPunctuator, build_tokenizer, PretrainedDataset, make_collate
)

def train_rubert(variant, epochs=10, lr=5e-5, batch_size=8):
    tok = build_tokenizer(variant)
    model = RuBertPunctuator(variant)
    ds = PretrainedDataset(texts, tok, max_len=128)
    cfn = make_collate(tok.pad_token_id)
    print(f"--- ruBERT [{variant}] ---")
    train_model(model, ds, cfn, epochs=epochs, batch_size=batch_size, lr=lr, device=DEVICE)
    return PunctuationRestorer(model, backend="rubert", tokenizer=tok, device=DEVICE)

# Раскомментируйте нужное (требует интернет):
# rubert_tiny = train_rubert("tiny", epochs=10)
# rubert_base = train_rubert("base", epochs=6, lr=3e-5)
print("Ячейка готова. Раскомментируйте вызовы train_rubert для запуска с интернетом.")

## 5. Демо-прогон: восстановление пунктуации

Подаём «сырой» текст (как от Whisper — нижний регистр, без знаков) и сравниваем модели.

In [ ]:
restorer_lstm = PunctuationRestorer(lstm, backend="word", vocab=vocab, device=DEVICE)
restorer_tr   = PunctuationRestorer(transformer, backend="word", vocab=vocab, device=DEVICE)

raw_inputs = [
    "москва столица россии а ты бывал в москве это потрясающий город",
    "сегодня отличная погода светит солнце поют птицы хочется гулять весь день",
    "он сказал я приду завтра но так и не пришёл какая досада",
]

for raw in raw_inputs:
    print("ВХОД   :", raw)
    print("LSTM   :", restorer_lstm(raw))
    print("TRANSF :", restorer_tr(raw))
    # При наличии обученного ruBERT:
    # print("ruBERT :", rubert_tiny(raw))
    print("-" * 70)

## 6. Встраивание в SpeechToText-пайплайн

`STTPunctuationPipeline` — это и есть **второе звено**. Принимает выход STT
(строку, `{"text": ...}` или `{"words": [...]}`) и возвращает текст с пунктуацией.

In [ ]:
pipeline = STTPunctuationPipeline(restorer_lstm)

# имитация разных форматов выхода Whisper
print(pipeline("утром я выпил кофе прочитал новости и пошёл на работу"))
print(pipeline({"text": "ты уверен я бы не стал так рисковать"}))
print(pipeline({"words": "какой прекрасный закат небо стало розовым".split()}))

### Подключение реального Whisper
```python
import whisper
asr = whisper.load_model("large-v3")
res = asr.transcribe("audio.wav", language="ru")

pipeline = STTPunctuationPipeline(rubert_tiny)   # любое из обученных звеньев
final_text = pipeline({"text": res["text"]})
print(final_text)
```
Так модель пунктуации становится вторым звеном: **звук → текст без пунктуации → текст с пунктуацией**.

## 7. Сохранение и загрузка
```python
# baseline
torch.save(lstm.state_dict(), "lstm_punct.pt")
vocab.save("vocab.json")

# ruBERT
torch.save(rubert_tiny.model.state_dict(), "rubert_tiny_punct.pt")
```
Загрузка: пересоздать модель тем же конструктором и вызвать `load_state_dict`;
для baseline дополнительно `WordVocab.load("vocab.json")`.